# Project 1.2: Xây dựng Chatbot RAG hỏi đáp tài liệu


## 1.Import thư viện


In [1]:
import pypdf  # Đọc nội dụng pdf
import chromadb  # Lưu trữ tìm kiếm vector (vector database)
import ollama  # Giao tiếp với server Ollama để tạo embedding và sinh văn bản

## 2. Xây dựng chương trình RAG


### 2.1. Đọc file pdf (Document loading)


In [ ]:
# Đọc nội dung file pdf
reader = pypdf.PdfReader("../documents/Valve_NewEmployeeHandbook.pdf")

# Ghép nội dung tất cả các trang thành 1 chuỗi text:
full_text = "\n".join(
    page.extract_text() or "" for page in reader.pages
)  # text trong mỗi trang dính lại bằng "\n"

print("Số trang: ", len(reader.pages))
print("Tổng số ký tự: ", len(full_text))
print(type(full_text))

Số trang:  37
Tổng số ký tự:  58352
<class 'str'>


### 2.2 Chunking


In [18]:
def chunk_text(text, size=1000, overlap=200):
    """
    Cắt text thành các đoạn nhỏ  có độ dài tối đa là 'size' ký tự,
    vời 'overlap' ký tự trùng lặp giữa 2 đoạn lệnh liên tiếp
    """
    paras = [p.strip() for p in text.split("\n") if p.strip()]
    chunks, cur = [], ""
    for p in paras:
        if len(cur) + len(p) + 1 <= size:
            cur += p + "\n"
        else:
            if cur:
                chunks.append(cur.strip())
            cur = (cur[-overlap:] + p + "\n") if overlap else (p + "\n")
    if cur.strip():
        chunks.append(cur.strip())
    return chunks


chunks = chunk_text(full_text)
print("Số chunks: ", len(chunks))
print()
print(chunks[0][:300])  # Xem 300 ký tự đầu tiên

Số chunks:  74

HANDBOOK FOR
NEW EMPLOYEES
HANDBOOK FOR
NEW EMPLOYEES
A fearless adventure
in knowing what to do
when no one’s there
telling you what to do
FIRST EDITION
2012
Dedicated to the famili


In [ ]:
text = " abc jdsd sad " + "\n" + "\n" + "123 45 6789   " + "\n" + "trái tim bên lề."
paras = [p.strip() for p in text.split("\n") if p.strip()]
print(paras)

['abc jdsd sad', '123 45 6789', 'trái tim bên lề.']


### 2.3 Embedding và lưu vào Vector database


In [ ]:
# Hàm tạo embedding từ danh sách text
def embed(texts):  # từ str thanh array
    """
    Chuyển danh sách chuổi text thành danh sách vector
    """
    return ollama.embed(model="bge-m3", input=texts)["embeddings"]


embed(chunks)
# Tạo vector database trong bộ nhớ
client = chromadb.Client()
collection = client.get_or_create_collection("rag")

# Thêm tất cả chunks vào database
collection.add(
    ids=[str(i) for i in range(len(chunks))],
    documents=chunks,
    embeddings=embed(chunks),
)

print("Đã index: ", collection.count(), "chunks")

Đã index:  74 chunks


### 2.4 Tìm đoạn liên quan (Retrieve)


In [ ]:
def retrieve(query, k=4):
    """
    Tìm k đoạn văn bản liên quan nhất với câu hỏi.
    """
    res = collection.query(
        query_embeddings=embed([query]), n_results=k
    )  # Mặc định so sánh bằng L2, nếu muốn chuyển qua cosine thì quay về collection chỉnh metadata={"hnsw:space":"cosine"}
    return res["documents"][0]


# Thử tìm kiếm
QUERY = "what should i do in the 1st day?"
for doc in retrieve(QUERY):
    print(doc[:200])
    print("-" * 40)

HANDBOOK FOR
NEW EMPLOYEES
HANDBOOK FOR
NEW EMPLOYEES
A fearless adventure
in knowing what to do
when no one’s there
telling you what to do
----------------------------------------
mescom in Cologne,
Germany, with the
ﬁrst annual Dota 2
championship.
What’s next? You tell us…
VALVE HFNE:10:11:12::08
– 19 –
SETTLING IN
issue with whomever you feel would help. Dina loves to force

----------------------------------------
......... 7
Your First Month
What to Work On
Why do I need to pick my own projects?, But how do I decide which things to
work on?, How do I find out what projects are under way?, Short-term vs. long-

----------------------------------------
First Day
So you’ve gone through the interview process, you’ve
signed the contracts, and you’re finally here at Valve.
Congratulations, and welcome.
Valve has an incredibly unique way of doing things

----------------------------------------


### 2.5 Hỏi đáp với LLM


In [23]:
# Mẫu prompt hướng dẫn LLM cách trả lời
PROMPT = """ Bạn là trợ lý hỏi đáp. Dùng các đoạn ngữ cảnh dưới đây để trả lời câu hỏi.
Nếu ngũ cảnh không có thông tin, hãy nói bạn không biết, đừng bịa.
Trả lời ngắn gọn, chính xác, bằng tiếng Việt.

Ngữ cảnh: {context}

Câu hỏi: {question}

Trả lời: """


def rag(question, k=4):
    """Hàm RAG chính: Tìm context rồi hỏi LLM."""
    context = "\n\n".join(retrieve(question, k))
    resp = ollama.chat(
        model="vicuna:7b-v1.5-q5_1",
        messages=[
            {
                "role": "user",
                "content": PROMPT.format(context=context, question=question),
            }
        ],
        options={"temperature": 0},
    )
    return resp["message"]["content"]


# Chạy thử
print(rag("Dota là gì?"))

Dota là một trò chơi đối kháng hoàn tất của Valve Corporation. Nó được phát triển và phát hành lần đầu tiên vào năm 2011 và đã trở thành một trong những trò chơi đối kháng phổ biến nhất trên toàn cầu. Trong trò chơi này, hai đội bóng sẽ giao chiến với nhau để kiểm soát một địa điểm hoặc đánh bại đối thủ. Các người chơi có thể chọn các nhân vật và trang bị đồng phục để tăng khả năng chiến đấu của mình. Dota 2 đã trở thành một phần quan trọng của văn hóa trò chơi và đã tạo ra nhiều giải đấu chuyên nghiệp lớn.
